In [1]:
# Maintenance done on 09/04/2025 @ 17:00
# JingyiWu

# Maintenance done on 29/06/2022 @ 8:30
# Gabin

#made by LN
#updated by LLZ
#on 04/11/2021

from bs4 import BeautifulSoup
import datetime
import numpy as np
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from time import sleep
import os
import re
import requests
import tabula
import pdfplumber
import camelot

In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'CH FINMA' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.2.0")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - moodys.com\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

Running CH FINMA Web Scraping Tool v.2.0


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()




# %%

In [ ]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict = { 
            'CH FINMA 1': 'https://www.finma.ch/en/~/media/finma/dokumente/bewilligungstraeger/xlsx/beh.xlsx?la=en', 
            'CH FINMA 2': 'https://www.finma.ch/en/~/media/finma/dokumente/bewilligungstraeger/xlsx/raiff.xlsx?la=en',
            'CH FINMA 3': 'https://www.finma.ch/en/~/media/finma/dokumente/bewilligungstraeger/xlsx/repbeh.xlsx?la=en',
            'CH FINMA 4': 'https://www.finma.ch/en/~/media/finma/dokumente/bewilligungstraeger/xlsx/beh_status2.xlsx?la=en',
            'CH FINMA 5': 'https://www.finma.ch/en/~/media/finma/dokumente/bewilligungstraeger/xlsx/vu.xlsx?la=en',
            'CH FINMA 6': 'https://www.finma.ch/en/~/media/finma/dokumente/bewilligungstraeger/xlsx/vk.xlsx?la=en',
            'CH FINMA 7': 'https://www.finma.ch/de/~/media/finma/dokumente/bewilligungstraeger/xlsx/fintech.xlsx?sc_lang=en',
            'CH FINMA 8': '', 
            # This list has 0   results in the register, as you could see. Please do not take this one into   account.
            # old link: https://www.finma.ch/de/~/media/finma/dokumente/finmapublic/bewilligungstraeger/versicherungsunternehmen-abkommen-fl-freier-dienstleistungsverkehr.pdf?la=de
            'CH FINMA 9': 'https://www.finma.ch/en/~/media/finma/dokumente/bewilligungstraeger/xlsx/afch.xlsx?la=en',
            'CH FINMA 10': 'https://www.finma.ch/en/~/media/finma/dokumente/bewilligungstraeger/xlsx/afetr.xlsx?la=en',
            'CH FINMA 11': 'https://www.finma.ch/en/~/media/finma/dokumente/bewilligungstraeger/xlsx/flvervt.xlsx?la=en',
            'CH FINMA 12': '',
            'CH FINMA 13': '',
            'CH FINMA 14': '',

            'CH FINMA 16': 'https://www.finma.ch/en/~/media/finma/dokumente/bewilligungstraeger/xlsx/vvtr.xlsx?la=en',
            'CH FINMA 17': 'https://www.finma.ch/en/~/media/finma/dokumente/bewilligungstraeger/xlsx/regst.xlsx?la=en',
            'CH FINMA 18': 'https://www.finma.ch/en/~/media/finma/dokumente/bewilligungstraeger/xlsx/repvkv.xlsx?la=en',
            'CH FINMA 19': 'https://www.finma.ch/en/~/media/finma/dokumente/bewilligungstraeger/xlsx/repvvtr.xlsx?la=en',
            'CH FINMA 20': 'https://www.finma.ch/en/~/media/finma/dokumente/bewilligungstraeger/xlsx/prprosp.xlsx?la=en',
            'CH FINMA 21': 'https://www.finma.ch/en/~/media/finma/dokumente/bewilligungstraeger/xlsx/sro.xlsx?sc_lang=en',
            'CH FINMA 22': 'https://www.finma.ch/en/~/media/finma/dokumente/bewilligungstraeger/xlsx/ao.xlsx?la=en',       
            
            #'CH FINMA 15': 'https://www.finma.ch/en/finma-public/warning-list/',     
          }

print('The current folder is: {}\nThe temp folder is: {}'.format(scriptfolder, tempfolder))



Typology={

        regulatorName+' 1': 'Authorised banks and securities dealers',
        regulatorName+' 2': 'Authorised Raiffeisen banks',
        regulatorName+' 3': 'Authorised representative offices of foreign banks and securities dealers',
        regulatorName+' 4': 'Banks and securities dealers discontinuing their business activities',
        regulatorName+' 5': 'Insurance companies under FINMA supervision',
        regulatorName+' 6': 'Insurance groups under FINMA supervision',
        regulatorName+' 7': 'Persons licensed by FINMA pursuant to Article 1b BA (FinTech licence)',
        regulatorName+' 8': 'Recognised foreign trading venues pursuant to the Federal Council Ordinance of 30 November 2018 on the Recognition of Foreign Trading Venues for the Trading of Equity Securities of Companies with Registered Office in Switzerland',

        regulatorName+' 9': 'Approved Swiss collective investment schemes',
        regulatorName+' 10': 'Foreign collective investment schemes authorised by FINMA for offering in Switzerland',
        regulatorName+' 11': 'Authorised fund management companies, representatives of foreign collective investment   schemes and distributors of collective investment schemes',
        regulatorName+' 12': '',
        regulatorName+' 13': '',
        regulatorName+' 14': '',
        regulatorName+' 15': 'Warning list',
        regulatorName+' 16': 'Portfolio managers and trustees licensed by FINMA and monitored by a supervisory organisation',
        
        regulatorName+' 17': 'Registration bodies licensed by FINMA',
        regulatorName+' 18': 'Representative offices of foreign managers of collective assets authorised by FINMA',
        regulatorName+' 19': 'Representative offices of foreign portfolio managers and trustees authorised by FINMA',
        regulatorName+' 20': 'Reviewing bodies for prospectuses licensed by FINMA',
        regulatorName+' 21': 'Self-regulatory organisations (SROs) recognised by FINMA',
        regulatorName+' 22': 'Supervisory organisations authorised by FINMA',

        }



sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

         'Phone - Mother company': [], 'Check': []}


now = datetime.datetime.now()

processdate = now.strftime('%Y-%m-%d')



ISO_EN = {"": "", 'NAN': '', 'OTHER': '', "AFGHANISTAN": "AF", "ÅLAND ISLANDS": "AX", "ALBANIA": "AL", "ALGERIA": "DZ", "AMERICAN SAMOA": "AS", "ANDORRA": "AD", "ANGOLA": "AO", "ANGUILLA": "AI", "ANTARCTICA": "AQ", "ANTIGUA AND BARBUDA": "AG", "ARGENTINA": "AR", "ARMENIA": "AM", "ARUBA": "AW", "AUSTRALIA": "AU", "AUSTRIA": "AT", "AZERBAIJAN": "AZ", "BAHAMAS, THE": "BS", "BAHRAIN": "BH", "BANGLADESH": "BD", "BARBADOS": "BB", "BELARUS": "BY", "BELGIUM": "BE", "BELIZE": "BZ", "BENIN": "BJ", "BERMUDA": "BM", "BHUTAN": "BT", "BOLIVIA": "BO", "BONAIRE, SINT EUSTATIUS AND SABA": "BQ", "BOSNIA AND HERZEGOVINA": "BA", "BOTSWANA": "BW", "BOUVET ISLAND": "BV", "BRAZIL": "BR", "BRITISH INDIAN OCEAN TERRITORY": "IO", "BRUNEI": "BN", "BULGARIA": "BG", "BURKINA FASO": "BF", "BURUNDI": "BI", "CABO VERDE": "CV", "CAMBODIA": "KH", "CAMEROON": "CM", "CANADA": "CA", "CAYMAN ISLANDS": "KY", "CENTRAL AFRICAN REPUBLIC": "CF", "CHAD": "TD", "CHILE": "CL", "CHINA, PEOPLES REPUBLIC OF": "CN","CHINA": "CN", "CHRISTMAS ISLAND": "CX", "COCOS (KEELING) ISLANDS": "CC", "COLOMBIA": "CO", "COMOROS": "KM", "CONGO": "CG", "CONGO, DEMOCRATIC REPUBLIC OF THE": "CD", "COOK ISLANDS": "CK", "COSTA RICA": "CR", "CÔTE D'IVOIRE": "CI", "CROATIA": "HR", "CUBA": "CU", "CURACAO, BONAIRE, SABA, ST. MARTIN & ST.": "CW", "CYPRUS": "CY", "CZECH REPUBLIC": "CZ", "DENMARK": "DK", "DJIBOUTI": "DJ", "DOMINICA": "DM", "DOMINICAN REPUBLIC": "DO", "ECUADOR": "EC", "EGYPT": "EG", "EL SALVADOR": "SV", "EQUATORIAL GUINEA": "GQ", "ERITREA": "ER", "ESTONIA": "EE", "ESWATINI": "SZ", "ETHIOPIA": "ET", "FALKLAND ISLANDS (MALVINAS)": "FK", "FAROE ISLANDS": "FO", "FIJI": "FJ", "FINLAND": "FI", "FRANCE": "FR", "FRENCH GUIANA": "GF", "FRENCH POLYNESIA": "PF", "FRENCH SOUTHERN TERRITORIES": "TF", "GABON": "GA", "GAMBIA": "GM", "GEORGIA": "GE", 'GEORGIA/GRUZINSKAYA': 'GE', "GERMANY": "DE", "GHANA": "GH", "GIBRALTAR": "GI", "GREECE": "GR", "GREENLAND": "GL", "GRENADA": "GD", "GUADELOUPE": "GP", "GUAM": "GU", "GUATEMALA": "GT", "GUERNSEY": "GG", "GUINEA": "GN", "GUINEA-BISSAU": "GW", "GUYANA": "GY", "HAITI": "HT", "HEARD ISLAND AND MCDONALD ISLANDS": "HM", "HOLY SEE": "VA", "HONDURAS": "HN", "HONG KONG": "HK", "HUNGARY": "HU", "ICELAND": "IS", "INDIA": "IN", "INDONESIA": "ID", "IRAN": "IR", "IRAQ": "IQ", "IRELAND": "IE", "ISLE OF MAN": "IM", "ISRAEL": "IL", "ITALY": "IT", "JAMAICA": "JM", "JAPAN": "JP", "JERSEY": "JE", "JORDAN": "JO", "KAZAKHSTAN": "KZ", "KENYA": "KE", "KIRIBATI": "KI", """KOREA (DEMOCRATIC PEOPLE"S REPUBLIC OF)""": "KP", "KOREA, SOUTH": "KR", "KUWAIT": "KW", "KYRGYZSTAN": "KG", "LAO PEOPLE'S DEMOCRATIC REPUBLIC": "LA", "LATVIA": "LV", "LEBANON": "LB", "LESOTHO": "LS", "LIBERIA": "LR", "LIBYA": "LY", "LIECHTENSTEIN": "LI", "LITHUANIA": "LT", "LUXEMBOURG": "LU", "MACAU": "MO", "MADAGASCAR": "MG", "MALAWI": "MW", "MALAYSIA": "MY", "MALDIVES": "MV", "MALI": "ML", "MALTA": "MT", "MARSHALL ISLANDS": "MH", "MARTINIQUE": "MQ", "MAURITANIA": "MR", "MAURITIUS": "MU", "MAYOTTE": "YT", "MEXICO": "MX", "FEDERATED STATES OF MICRONESIA": "FM", "MOLDOVA, REPUBLIC OF": "MD", "MONACO": "MC", "MONGOLIA": "MN", "MONTENEGRO": "ME", "MONTSERRAT": "MS", "MOROCCO": "MA", "MOZAMBIQUE": "MZ", "MYANMAR": "MM", "NAMIBIA": "NA", "NAURU": "NR", "NEPAL": "NP", "NETHERLANDS": "NL", "NEW CALEDONIA": "NC", "NEW ZEALAND": "NZ", "NICARAGUA": "NI", "NIGER": "NE", "NIGERIA": "NG", "NIUE": "NU", "NORFOLK ISLAND": "NF", "NORTH MACEDONIA": "MK", "NORTHERN MARIANA ISLANDS": "MP", "NORWAY": "NO", "OMAN": "OM", "PAKISTAN": "PK", "PALAU": "PW", "PALESTINE, STATE OF": "PS", "PANAMA": "PA", "PAPUA NEW GUINEA": "PG", "PARAGUAY": "PY", "PERU": "PE", "PHILIPPINES": "PH", "PITCAIRN": "PN", "POLAND": "PL", "PORTUGAL": "PT", "PUERTO RICO": "PR", "QATAR": "QA", "RÉUNION": "RE", "ROMANIA": "RO", "RUSSIA": "RU", "RWANDA": "RW", "SAINT BARTHÉLEMY": "BL", "SAINT HELENA, ASCENSION AND TRISTAN DA CUNHA": "SH", "SAINT KITTS AND NEVIS": "KN", "SAINT LUCIA": "LC", "SAINT MARTIN (FRENCH PART)": "MF", "SAINT PIERRE AND MIQUELON": "PM", "SAINT VINCENT AND THE GRENADINES": "VC", "SAMOA": "WS", "SAN MARINO": "SM", "SAO TOME AND PRINCIPE": "ST", "SAUDI ARABIA": "SA", "SENEGAL": "SN", "SERBIA": "RS", "SEYCHELLES": "SC", "SIERRA LEONE": "SL", "SINGAPORE": "SG", "SINT MAARTEN (DUTCH PART)": "SX", "SLOVAKIA": "SK", "SLOVAK REPUBLIC": "SK", "SLOVENIA": "SI", "SOLOMON ISLANDS": "SB", "SOMALIA": "SO", "SOUTH AFRICA": "ZA", "SOUTH GEORGIA AND THE SOUTH SANDWICH ISLANDS": "GS", "SOUTH SUDAN": "SS", "SPAIN": "ES", "SRI LANKA": "LK", "SUDAN": "SD", "SURINAME": "SR", "SVALBARD AND JAN MAYEN": "SJ", "SWEDEN": "SE", "SWITZERLAND": "CH", "SYRIAN ARAB REPUBLIC": "SY", "TAIWAN": "TW",'TAIWAN,  REPUBLIC OF CHINA': 'TW' ,"TAJIKISTAN": "TJ", "TANZANIA, UNITED REPUBLIC OF": "TZ", "THAILAND": "TH", "TIMOR-LESTE": "TL", "TOGO": "TG", "TOKELAU": "TK", "TONGA": "TO", "TRINIDAD AND TOBAGO": "TT", "TUNISIA": "TN", "TURKEY": "TR", "TURKMENISTAN": "TM", "TURKS & CAICOS ISLANDS": "TC", "TUVALU": "TV", "UGANDA": "UG", "UKRAINE": "UA", "UNITED ARAB EMIRATES": "AE", "UNITED KINGDOM OF GREAT BRITAIN AND NORTHERN IRELAND": "GB", "UNITED STATES": "US", "UNITED STATES MINOR OUTLYING ISLANDS": "UM", "URUGUAY": "UY", "UZBEKISTAN": "UZ", "VANUATU": "VU", "VENEZUELA": "VE", "VIETNAM": "VN", "BRITISH VIRGIN ISLANDS": "VG", "VIRGIN ISLANDS OF THE U.S.": "VI", "WALLIS AND FUTUNA": "WF", "WESTERN SAHARA": "EH", "YEMEN": "YE", "ZAMBIA": "ZM", "ZIMBABWE": "ZW", "ENGLAND": "GB", "UNITED KINGDOM": "GB", "UNITED KINGDOM (OTHER)": "GB", "FRANCE (OTHER)": "FR", "WALES": "GB", "CONGO (KINSHASA)": "CD", "CONGO (BRAZZAVILLE)": "CD", "SCOTLAND": "GB", "ITALY (OTHER)": "IT", "INDONESIA (OTHER)": "ID", "INDIA (OTHER)": "IN", "MOROCCO (OTHER)": "MA", "NEW ZEALAND (OTHER)": "NZ", "SWITZERLAND (OTHER)": "CH", "MALAYSIA (OTHER)": "MY", "NETHERLANDS ANTILLES": "AN", "TRINIDAD & TOBAGO (OTHER)": "TT", "CHANNEL ISLANDS": "GB", "UNITED ARAB EMIRATES (OTHER)": "AE", "DENMARK (OTHER)": "DK", "COMORO ISLANDS": "KM", "MACEDONIA (FORMER YUGOSLAV REPUBLIC OF)": "MK", "SERBIA AND MONTENEGRO(FORMER YUGOSLAVIA)": "CS", "TRINIDAD": "TT", "ETHIOPIA (OTHER)": "ET", "IVORY COAST": "CI", "DUBAI": "AE", "BRITISH WEST INDIES (OTHER)": "VG", "SWAZILAND": "SZ", 'UNITED KINGDOM  (OTHER)': 'GB'}
ISO_DE = {"": "", 'NAN': '', 'OTHER': '', "AFGHANISTAN": "AF", "ÅLAND ISLANDS": "AX", "ALBANIEN": "AL", "ALGERIEN": "DZ", "AMERIKANISCH-SAMOA": "AS", "ANDORRA": "AD", "ANGOLA": "AO", "ANGUILLA": "AI", "ANTARKTIS": "AQ", "ANTIGUA UND BARBUDA": "AG", "ARGENTINIEN": "AR", "ARMENIEN": "AM", "ARUBA": "AW", "AUSTRALIEN": "AU", "ÖSTERREICH": "AT", "ASERBAIDSCHAN": "AZ", "BAHAMAS, THE": "BS", "BAHRAIN": "BH", "BANGLADESCH": "BD", "BARBADOS": "BB", "WEIßRUSSLAND": "BY", "BELGIEN": "BE", "BELIZE": "BZ", "BENIN": "BJ", "BERMUDA": "BM", "BHUTAN": "BT", "BOLIVIEN": "BO", "BONAIRE, SINT EUSTATIUS AND SABA": "BQ", "BOSNIEN UND HERZEGOWINA": "BA", "BOTSUANA": "BW", "BOUVET ISLAND": "BV", "BRASILIEN": "BR", "BRITISH INDIAN OCEAN TERRITORY": "IO", "BRUNEI": "BN", "BULGARIEN": "BG", "BURKINA FASO": "BF", "BURUNDI": "BI", "KAP VERDE": "CV", "KAMBODSCHA": "KH", "KAMERUN": "CM", "KANADA": "CA", "CAYMANINSELN, AUCH KAIMANINSELN": "KY", "ZENTRALAFRIKANISCHE REPUBLIK": "CF", "TSCHAD": "TD", "CHILE": "CL", "CHINA, VOLKSREPUBLIK CHINA": "CN","CHINA": "CN", "WEIHNACHTSINSEL": "CX", "KOKOSINSELN, KEELINGINSELN": "CC", "KOLUMBIEN": "CO", "KOMOREN": "KM", "REPUBLIK KONGO": "CG", "REPUBLIK KONGO": "CD", "COOKINSELN": "CK", "COSTA RICA": "CR", "ELFENBEINKÜSTE": "CI", "KROATIEN": "HR", "KUBA": "CU", "CURACAO, BONAIRE, SABA, ST. MARTIN & ST.": "CW", "ZYPERN": "CY", "TSCHECHIEN": "CZ", "DÄNEMARK": "DK", "DSCHIBUTI": "DJ", "DOMINICA": "DM", "DOMINIKANISCHE REPUBLIK": "DO", "ECUADOR": "EC", "ÄGYPTEN": "EG", "EL SALVADOR": "SV", "ÄQUATORIALGUINEA": "GQ", "ERITREA": "ER", "ESTLAND": "EE", "ESWATINI": "SZ", "ÄTHIOPIEN": "ET", "FALKLANDINSELN": "FK", "FÄRÖER-INSELN": "FO", "FIDSCHI": "FJ", "FINNLAND": "FI", "FRANKREICH": "FR", "FRANZÖSISCH-GUAYANA": "GF", "FRANZÖSISCH-POLYNESIEN": "PF", "FRANZÖSISCHE SÜD- UND ANTARKTISGEBIETE": "TF", "GABUN": "GA", "GAMBIA": "GM", "GEORGIEN": "GE", 'GEORGIA/GRUZINSKAYA': 'GE', "DEUTSCHLAND": "DE", "GHANA": "GH", "GIBRALTAR": "GI", "GRIECHENLAND": "GR", "GRÖNLAND": "GL", "GRENADA": "GD", "GUADELOUPE": "GP", "GUAM": "GU", "GUATEMALA": "GT", "GUERNSEY": "GG", "GUINEA": "GN", "GUINEA-BISSAU": "GW", "GUYANA": "GY", "HAITI": "HT", "HEARD ISLAND AND MCDONALD ISLANDS": "HM", "HOLY SEE": "VA", "HONDURAS": "HN", "HONG KONG": "HK", "UNGARN": "HU", "ISLAND": "IS", "INDIEN": "IN", "INDONESIEN": "ID", "IRAN": "IR", "IRAK": "IQ", "IRLAND": "IE", "ISLE OF MAN": "IM", "ISRAEL": "IL", "ITALIEN": "IT", "JAMAICA": "JM", "JAPAN": "JP", "JERSEY": "JE", "JORDANIEN": "JO", "KASACHSTAN": "KZ", "KENIA": "KE", "KIRIBATI": "KI", "NORDKOREA": "KP", "SÜDKOREA": "KR", "KUWAIT": "KW", "KIRGISISTAN": "KG", "LAOS": "LA", "LETTLAND": "LV", "LIBANON": "LB", "LESOTHO": "LS", "LIBERIA": "LR", "LIBYEN": "LY", "LIECHTENSTEIN": "LI", "LITAUEN": "LT", "LUXEMBURG": "LU", "MACAU": "MO", "MADAGASKAR": "MG", "MALAWI": "MW", "MALAYSIA": "MY", "MALEDIVEN": "MV", "MALI": "ML", "MALTA": "MT", "MARSHALLINSELN": "MH", "MARTINIQUE": "MQ", "MAURITANIEN": "MR", "MAURITIUS": "MU", "MAYOTTE": "YT", "MEXICO": "MX", "FÖDERIERTE STAATEN VON MIKRONESIEN": "FM", "REPUBLIK MOLDAU": "MD", "MONACO": "MC", "MONGOLEI": "MN", "MONTENEGRO": "ME", "MONTSERRAT": "MS", "MAROKKO": "MA", "MOSAMBIK": "MZ", "MYANMAR": "MM", "NAMIBIA": "NA", "NAURU": "NR", "NEPAL": "NP", "NIEDERLANDE": "NL", "NEUKALEDONIEN": "NC", "NEUSEELAND": "NZ", "NICARAGUA": "NI", "NIGER": "NE", "NIGERIA": "NG", "NIUE": "NU", "NORDMAZEDONIEN": "MK", "NÖRDLICHE MARIANEN": "MP", "NORWEGEN": "NO", "OMAN": "OM", "PAKISTAN": "PK", "PALAU": "PW", "STAAT PALÄSTINA": "PS", "PANAMA": "PA", "PAPUA-NEUGUINEA": "PG", "PARAGUAY": "PY", "PERU": "PE", "PHILIPPINEN": "PH", "PITCAIRN": "PN", "POLEN": "PL", "PORTUGAL": "PT", "PUERTO RICO": "PR", "KATAR": "QA", "RÉUNION": "RE", "RUMÄNIEN": "RO", "RUSSLAND": "RU", "RUANDA": "RW", "ST KITTS UND NEVIS": "KN", "ST. LUCIA": "LC", "ST. VINCENT UND DIE GRENADINEN": "VC", "SAMOA": "WS", "SAN MARINO": "SM", "SAO TOME UND PRINCIPE": "ST", "SAUDI-ARABIEN": "SA", "SENEGAL": "SN", "SERBIEN": "RS", "SEYCHELLEN": "SC", "SIERRA LEONE": "SL", "SINGAPUR": "SG", "SLOWAKEI": "SK", "SLOVENIA": "SI", "SALOMONEN": "SB", "SOMALIA": "SO", "SÜDAFRIKA": "ZA", "SÜDSUDAN": "SS", "SPANIEN": "ES", "SRI LANKA": "LK", "SUDAN": "SD", "SURINAME": "SR", "SCHWEDEN": "SE", "SCHWEIZ": "CH", "SYRIEN": "SY", "TAIWAN": "TW",'TAIWAN,  REPUBLIK CHINA': 'TW' ,"TADSCHIKISTAN": "TJ", "TANSANIA": "TZ", "THAILAND": "TH", "OSTTIMOR": "TL", "TOGO": "TG", "TOKELAU": "TK", "TONGA": "TO", "TRINIDAD UND TOBAGO": "TT", "TUNISIEN": "TN", "TÜRKEI": "TR", "TURKMENISTAN": "TM", "TURKS- UND CAICOSINSELN": "TC", "TUVALU": "TV", "UGANDA": "UG", "UKRAINE": "UA", "VEREINIGTE ARABISCHE EMIRATE": "AE", "VEREINIGTES KÖNIGREICH": "GB", "VEREINIGTE STAATEN": "US", "URUGUAY": "UY", "USBEKISTAN": "UZ", "VANUATU": "VU", "VENEZUELA": "VE", "VIETNAM": "VN", "BRITISCHE JUNGFERNINSELN": "VG", "WALLIS UND FUTUNA": "WF", "WESTSAHARA": "EH", "JEMEN": "YE", "SAMBIA": "ZM", "SIMBABWE": "ZW", "ENGLAND": "GB", "VEREINIGTES KÖNIGREICH": "GB", "TRINIDAD": "TT", "ELFENBEINKÜSTE": "CI"}
ISO_FR = {"": "", 'NAN': '', 'OTHER': '', 'AFGHANISTAN' : 'AF', 'AFRIQUE DU SUD' : 'ZA', 'ÅLAND, ÎLES' : 'AX', 'ALBANIE' : 'AL', 'ALGÉRIE' : 'DZ', 'ALLEMAGNE' : 'DE', 'ANDORRE' : 'AD', 'ANGOLA' : 'AO', 'ANGUILLA' : 'AI', 'ANTARCTIQUE' : 'AQ', 'ANTIGUA-ET-BARBUDA' : 'AG', 'ARABIE SAOUDITE' : 'SA', 'ARGENTINE' : 'AR', 'ARMÉNIE' : 'AM', 'ARUBA' : 'AW', 'AUSTRALIE' : 'AU', 'AUTRICHE' : 'AT', 'AZERBAÏDJAN' : 'AZ', 'BAHAMAS' : 'BS', 'BAHREÏN' : 'BH', 'BANGLADESH' : 'BD', 'BARBADE' : 'BB', 'BÉLARUS' : 'BY', 'BELGIQUE' : 'BE', 'BELIZE' : 'BZ', 'BÉNIN' : 'BJ', 'BERMUDES' : 'BM', 'BHOUTAN' : 'BT', 'BOLIVIE, L\'ÉTAT PLURINATIONAL DE' : 'BO', 'BONAIRE, SAINT-EUSTACHE ET SABA' : 'BQ', 'BOSNIE-HERZÉGOVINE' : 'BA', 'BOTSWANA' : 'BW', 'BOUVET, ÎLE' : 'BV', 'BRÉSIL' : 'BR', 'BRUNEI DARUSSALAM' : 'BN', 'BULGARIE' : 'BG', 'BURKINA FASO' : 'BF', 'BURUNDI' : 'BI', 'CAÏMANS, ÎLES' : 'KY', 'CAMBODGE' : 'KH', 'CAMEROUN' : 'CM', 'CANADA' : 'CA', 'CAP-VERT' : 'CV', 'CENTRAFRICAINE, RÉPUBLIQUE' : 'CF', 'CHILI' : 'CL', 'CHINE' : 'CN', 'CHRISTMAS, ÎLE' : 'CX', 'CHYPRE' : 'CY', 'COCOS (KEELING), ÎLES' : 'CC', 'COLOMBIE' : 'CO', 'COMORES' : 'KM', 'CONGO' : 'CG', 'CONGO, LA RÉPUBLIQUE DÉMOCRATIQUE DU' : 'CD', 'COOK, ÎLES' : 'CK', 'CORÉE, RÉPUBLIQUE DE' : 'KR', 'CORÉE, RÉPUBLIQUE POPULAIRE DÉMOCRATIQUE DE' : 'KP', 'COSTA RICA' : 'CR', 'CÔTE D\'IVOIRE' : 'CI', 'CROATIE' : 'HR', 'CUBA' : 'CU', 'CURAÇAO' : 'CW', 'DANEMARK' : 'DK', 'DJIBOUTI' : 'DJ', 'DOMINICAINE, RÉPUBLIQUE' : 'DO', 'DOMINIQUE' : 'DM', 'ÉGYPTE' : 'EG', 'EL SALVADOR' : 'SV', 'ÉMIRATS ARABES UNIS' : 'AE', 'ÉQUATEUR' : 'EC', 'ÉRYTHRÉE' : 'ER', 'ESPAGNE' : 'ES', 'ESTONIE' : 'EE', 'ÉTATS-UNIS' : 'US', 'ÉTHIOPIE' : 'ET', 'FALKLAND, ÎLES (MALVINAS)' : 'FK', 'FÉROÉ, ÎLES' : 'FO', 'FIDJI' : 'FJ', 'FINLANDE' : 'FI', 'FRANCE' : 'FR', 'GABON' : 'GA', 'GAMBIE' : 'GM', 'GÉORGIE' : 'GE', 'GÉORGIE DU SUD-ET-LES ÎLES SANDWICH DU SUD' : 'GS', 'GHANA' : 'GH', 'GIBRALTAR' : 'GI', 'GRÈCE' : 'GR', 'GRENADE' : 'GD', 'GROENLAND' : 'GL', 'GUADELOUPE' : 'GP', 'GUAM' : 'GU', 'GUATEMALA' : 'GT', 'GUERNESEY' : 'GG', 'GUINÉE' : 'GN', 'GUINÉE-BISSAU' : 'GW', 'GUINÉE ÉQUATORIALE' : 'GQ', 'GUYANA' : 'GY', 'GUYANE FRANÇAISE' : 'GF', 'HAÏTI' : 'HT', 'HEARD-ET-ÎLES MACDONALD, ÎLE' : 'HM', 'HONDURAS' : 'HN', 'HONG KONG' : 'HK', 'HONGRIE' : 'HU', 'ÎLE DE MAN' : 'IM', 'ÎLES MINEURES ÉLOIGNÉES DES ÉTATS-UNIS' : 'UM', 'ÎLES VIERGES BRITANNIQUES' : 'VG', 'ÎLES VIERGES DES ÉTATS-UNIS' : 'VI', 'INDE' : 'IN', 'INDONÉSIE' : 'ID', 'IRAN, RÉPUBLIQUE ISLAMIQUE D\'' : 'IR', 'IRAQ' : 'IQ', 'IRLANDE' : 'IE', 'ISLANDE' : 'IS', 'ISRAËL' : 'IL', 'ITALIE' : 'IT', 'JAMAÏQUE' : 'JM', 'JAPON' : 'JP', 'JERSEY' : 'JE', 'JORDANIE' : 'JO', 'KAZAKHSTAN' : 'KZ', 'KENYA' : 'KE', 'KIRGHIZISTAN' : 'KG', 'KIRIBATI' : 'KI', 'KOWEÏT' : 'KW', 'LAO, RÉPUBLIQUE DÉMOCRATIQUE POPULAIRE' : 'LA', 'LESOTHO' : 'LS', 'LETTONIE' : 'LV', 'LIBAN' : 'LB', 'LIBÉRIA' : 'LR', 'LIBYE' : 'LY', 'LIECHTENSTEIN' : 'LI', 'LITUANIE' : 'LT', 'LUXEMBOURG' : 'LU', 'MACAO' : 'MO', 'MACÉDOINE, L\'EX-RÉPUBLIQUE YOUGOSLAVE DE' : 'MK', 'MADAGASCAR' : 'MG', 'MALAISIE' : 'MY', 'MALAWI' : 'MW', 'MALDIVES' : 'MV', 'MALI' : 'ML', 'MALTE' : 'MT', 'MARIANNES DU NORD, ÎLES' : 'MP', 'MAROC' : 'MA', 'MARSHALL, ÎLES' : 'MH', 'MARTINIQUE' : 'MQ', 'MAURICE' : 'MU', 'MAURITANIE' : 'MR', 'MAYOTTE' : 'YT', 'MEXIQUE' : 'MX', 'MICRONÉSIE, ÉTATS FÉDÉRÉS DE' : 'FM', 'MOLDOVA, RÉPUBLIQUE DE' : 'MD', 'MONACO' : 'MC', 'MONGOLIE' : 'MN', 'MONTÉNÉGRO' : 'ME', 'MONTSERRAT' : 'MS', 'MOZAMBIQUE' : 'MZ', 'MYANMAR' : 'MM', 'NAMIBIE' : 'NA', 'NAURU' : 'NR', 'NÉPAL' : 'NP', 'NICARAGUA' : 'NI', 'NIGER' : 'NE', 'NIGÉRIA' : 'NG', 'NIUÉ' : 'NU', 'NORFOLK, ÎLE' : 'NF', 'NORVÈGE' : 'NO', 'NOUVELLE-CALÉDONIE' : 'NC', 'NOUVELLE-ZÉLANDE' : 'NZ', 'OCÉAN INDIEN, TERRITOIRE BRITANNIQUE DE L\'' : 'IO', 'OMAN' : 'OM', 'OUGANDA' : 'UG', 'OUZBÉKISTAN' : 'UZ', 'PAKISTAN' : 'PK', 'PALAOS' : 'PW', 'PALESTINIEN OCCUPÉ, TERRITOIRE' : 'PS', 'PANAMA' : 'PA', 'PAPOUASIE-NOUVELLE-GUINÉE' : 'PG', 'PARAGUAY' : 'PY', 'PAYS-BAS' : 'NL', 'PÉROU' : 'PE', 'PHILIPPINES' : 'PH', 'PITCAIRN' : 'PN', 'POLOGNE' : 'PL', 'POLYNÉSIE FRANÇAISE' : 'PF', 'PORTO RICO' : 'PR', 'PORTUGAL' : 'PT', 'QATAR' : 'QA', 'RÉUNION' : 'RE', 'ROUMANIE' : 'RO', 'ROYAUME-UNI' : 'GB', 'RUSSIE, FÉDÉRATION DE' : 'RU', 'RWANDA' : 'RW', 'SAHARA OCCIDENTAL' : 'EH', 'SAINT-BARTHÉLEMY' : 'BL', 'SAINTE-HÉLÈNE, ASCENSION ET TRISTAN DA CUNHA' : 'SH', 'SAINTE-LUCIE' : 'LC', 'SAINT-KITTS-ET-NEVIS' : 'KN', 'SAINT-MARIN' : 'SM', 'SAINT-MARTIN(PARTIE FRANÇAISE)' : 'MF', 'SAINT-MARTIN (PARTIE NÉERLANDAISE)' : 'SX', 'SAINT-PIERRE-ET-MIQUELON' : 'PM', 'SAINT-SIÈGE (ÉTAT DE LA CITÉ DU VATICAN)' : 'VA', 'SAINT-VINCENT-ET-LES GRENADINES' : 'VC', 'SALOMON, ÎLES' : 'SB', 'SAMOA' : 'WS', 'SAMOA AMÉRICAINES' : 'AS', 'SAO TOMÉ-ET-PRINCIPE' : 'ST', 'SÉNÉGAL' : 'SN', 'SERBIE' : 'RS', 'SEYCHELLES' : 'SC', 'SIERRA LEONE' : 'SL', 'SINGAPOUR' : 'SG', 'SLOVAQUIE' : 'SK', 'SLOVÉNIE' : 'SI', 'SOMALIE' : 'SO', 'SOUDAN' : 'SD', 'SOUDAN DU SUD' : 'SS', 'SRI LANKA' : 'LK', 'SUÈDE' : 'SE', 'SUISSE' : 'CH', 'SURINAME' : 'SR', 'SVALBARD ET ÎLE JAN MAYEN' : 'SJ', 'SWAZILAND' : 'SZ', 'SYRIENNE, RÉPUBLIQUE ARABE' : 'SY', 'TADJIKISTAN' : 'TJ', 'TAÏWAN, PROVINCE DE CHINE' : 'TW', 'TANZANIE, RÉPUBLIQUE-UNIE DE' : 'TZ', 'TCHAD' : 'TD', 'TCHÈQUE, RÉPUBLIQUE' : 'CZ', 'TERRES AUSTRALES FRANÇAISES' : 'TF', 'THAÏLANDE' : 'TH', 'TIMOR-LESTE' : 'TL', 'TOGO' : 'TG', 'TOKELAU' : 'TK', 'TONGA' : 'TO', 'TRINITÉ-ET-TOBAGO' : 'TT', 'TUNISIE' : 'TN', 'TURKMÉNISTAN' : 'TM', 'TURKS-ET-CAÏCOS, ÎLES' : 'TC', 'TURQUIE' : 'TR', 'TUVALU' : 'TV', 'UKRAINE' : 'UA', 'URUGUAY' : 'UY', 'VANUATU' : 'VU', 'VENEZUELA, RÉPUBLIQUE BOLIVARIENNE DU' : 'VE', 'VIET NAM' : 'VN', 'WALLIS ET FUTUNA' : 'WF', 'YÉMEN' : 'YE', 'ZAMBIE' : 'ZM', 'ZIMBABWE' : 'ZW'}

try:
    os.mkdir(tempfolder)
except:
    prevfiles=os.listdir(tempfolder)
    os.chdir(tempfolder)
    for prf in prevfiles:
        os.remove(prf)
    print('The directory tempfolder already exists.')
os.chdir(tempfolder)##only if files are going to be downloaded here

pattern = re.compile('([0-9]+)')



The current folder is: C:\Users\wuj1\OneDrive - moodys.com\Desktop\Regulator\CH FINMA
The temp folder is: C:\Users\wuj1\OneDrive - moodys.com\Desktop\Regulator\CH FINMA\tempfolder
The directory tempfolder already exists.


In [5]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict

def scroll_to_bottom(driver):
    # Get scroll height

    last_height = driver.execute_script("return document.body.scrollHeight")

    while True:
        # Scroll down to the bottom

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

        # Wait to load the page

        sleep(2)

        # Calculate new scroll height and compare with last scroll height

        new_height = driver.execute_script("return document.body.scrollHeight")

        if new_height == last_height:

            break
        last_height = new_height

def countryToISO(country):
    if country.strip().upper() in ISO_EN:
        crty_out = ISO_EN[country.strip().upper()]
    elif country.strip().upper() in ISO_DE:
        crty_out = ISO_DE[country.strip().upper()]
    elif country.strip().upper() in ISO_FR:
        crty_out = ISO_FR[country.strip().upper()]
    else:
        crty_out = ''
    return crty_out

In [6]:

# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for reg in regdict:
    print('Working with {}'.format(reg))
    if len(regdict[reg])==0:
        print(reg, '- DISREGARD')
        continue
#     elif len(regdict[reg])>5:
#         url = regdict[reg]
#         regutype = 'Authorised'
    else:
        url = regdict[reg]
        
    if '.xlsx' not in url.lower() and '.pdf' not in url.lower():
        driver.get(url)
        sleep(10)
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        table = soup.find('table')
        trs = table.find('tbody').find_all('tr')
        cdates = [tr.find_all('td')[-1].text.strip() for tr in trs] #date with dd.mm.yyy format
#         cdates = [ele.split('.')[-1]+'-'+ele.split('.')[1]+'-'+ele.split('.')[0] for ele in cdates] #date with yyyy-mm-dd format
        a_s = [tr.find('a', href=True) for tr in trs]
        names = [ele.text.strip() for ele in a_s]
        urls = ['https://www.finma.ch'+ele['href'] for ele in a_s]
        for rang in range(len(a_s)):
            #print(reg, f"- Working with {names[rang]}. {rang+1} out of {len(a_s)}")
            driver.get(urls[rang])
            sleep(0.5)
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            sqldict['Name'].append(names[rang])
            sqldict['CancellationDate'].append(cdates[rang])
            div = soup.find('div', {'class':'mod mod-content'})
            if div is not None and 'The requested page could not be found' not in div.text:
                trs = div.find_all('tr')
                contact_dict = dict()
                for tr in trs:
                    contact_dict[tr.find('th').text.strip()] = tr.find('td').text.strip()
                if len(contact_dict['Domicile'])>=len(contact_dict['Address']):
                    sqldict['Address_1'].append(contact_dict['Domicile'])
                else:
                    sqldict['Address_1'].append(contact_dict['Address'])
                sqldict['Website'].append(contact_dict['Internet'])
            sqldict['ListProcessDate'].append(processdate)
            sqldict['Cntry'].append('CH')
            sqldict['RegCtry'].append('CH')
            sqldict['RegCode'].append('FINMA')
            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict['RegulationType'].append("Unauthorised")
            # Fill the rest of regdict with empty string
            for key in sqldict:
                if len(sqldict['Name'])>len(sqldict[key]):
                    sqldict[key].append('')
                    
    elif '.xlsx' in url.lower():
        driver.get(url)
        sleep(3)
        
        while len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele])==0:
            print('Waiting for file to download')
            sleep(2)
            
        print(os.listdir(tempfolder))
        xlsx_file = os.listdir(tempfolder)[0]
        filePath = os.path.join(tempfolder,xlsx_file)
        if reg.split()[-1] in ['9', '10']:
            df_excel = pd.read_excel(filePath, header=None, dtype=str)
            df_excel.replace('*nan*', np.nan)
            os.remove(filePath)
            
            gate = False
            no_country = False
            
            for i in range(len(df_excel)):
                # condition to stop recording the names
                if gate and not df_excel[0][i] == df_excel[0][i]:
                    gate = False
            
                # the gate is open. Record the names
                if gate and df_excel[0][i].strip() != '':
                    #print('Name: ',df_excel[0][i].strip())
                    #print('FINMA-ID: ',df_excel[idx_id][i],'\n')
                    sqldict['Name'].append(df_excel[0][i].strip())
                    sqldict['InternalID_1'].append(df_excel[idx_id][i])
                    sqldict['InternalID_1_type'].append('FINMA-ID')
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegCtry'].append('CH')
                    sqldict['RegCode'].append('FINMA')
                    sqldict['ListCode'].append(reg.split()[-1])
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['ListName'].append(Typology[reg])
                    
                    if not no_country:
                        #print('Country: ',countryToISO(df_excel[idx_country][i]))
                        sqldict['Cntry'].append(countryToISO(df_excel[idx_country][i]))
                    else:
                        sqldict['Cntry'].append('CH')
                        
                    # Fill the rest of regdict with empty string
                    for key in sqldict:
                        if len(sqldict['Name'])>len(sqldict[key]):
                            sqldict[key].append('')
            
                # condition to start recording the names
                if df_excel[0][i] == df_excel[0][i] and df_excel[0][i].strip() in ['Name','Name of individual fund or sub-fund']:
                    gate = True
                    idx_id = df_excel.loc[i,:].tolist().index('FINMA-ID')
                    if 'Country' in df_excel.loc[i,:].tolist():
                        idx_country = df_excel.loc[i,:].tolist().index('Country')
                    else:
                        no_country = True
            
        else:
            df_excel = pd.read_excel(filePath, engine='openpyxl', header=None, dtype=str)
            df_excel.fillna('*nan*', inplace=True)
            os.remove(filePath)
        
            gate = False
            city_in_column_0 = False
            city_in_tuple_idx1 = False
            city_in_tuple_idx2 = False
        
            list_of_tuples = df_excel.to_numpy()#list_of_tuples = df_excel.index.values.tolist()
            #list_1 = []
            #list_2 = []
            
            # for el in list_of_tuples:
            #     if type(el) == tuple:
            #         list_1.append(el[1])
            #         list_2.append(el[2])
                
            if 'City' in df_excel[0].tolist():
                city_in_column_0 = True
            elif 'City' in [tuple(item)[1] for item in list_of_tuples]:
                city_in_tuple_idx1 = True
            elif 'City' in [tuple(item)[2] for item in list_of_tuples]:
                city_in_tuple_idx2 = True
            
            for i, tp in enumerate (list_of_tuples):
                nan_count = sum(x == '*nan*' for x in tp)
                if nan_count+1>=len(tp) and reg.split()[-1] != '6':
                    pass
                elif reg.split()[-1] == '6' and nan_count>=len(tp) and 'Total' in tp[0].strip():
                    pass
                else:
                # cast tp to tuple because of the python version!!
                    tp = tuple(tp)
                    # condition to stop recording the names
                    if gate and not tp[0] == tp[0] or '*nan*' in tp[0].strip() :
                        gate = False
                    # the gate is open. Record the names
                    if gate and tp[0].strip() != '' :
                        #print('Name: ',tp[0].strip())
                        sqldict['Name'].append(tp[0].strip())
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['Cntry'].append('CH')
                        sqldict['RegCtry'].append('CH')
                        sqldict['RegCode'].append('FINMA')
                        sqldict['ListCode'].append(reg.split()[-1])
                        sqldict['ListName'].append(Typology[reg])
                        if reg.split()[-1] == '4':
                            sqldict['RegulationType'].append('Regulated')
                        else:
                            sqldict['RegulationType'].append('Regulated')
                        if city_in_tuple_idx1 and tp[1] == tp[1] and tp[1].strip() != '':
                            #print('City: ',tp[1].strip())
                            sqldict['City'].append(tp[1].strip())
                        elif city_in_tuple_idx2 and tp[2] == tp[2] and tp[2].strip() != '':
                            #print('City: ',tp[2].strip())
                            sqldict['City'].append(tp[2].strip())
                        elif city_in_column_0 and df_excel[0][i] == df_excel[0][i] and df_excel[0][i].strip() != '':
                            #print('City: ',df_excel[0][i].strip())
                            sqldict['City'].append(df_excel[0][i].strip())
                        else:
                            #print('City: ','')
                            sqldict['City'].append('')
                            
                        # Fill the rest of regdict with empty string
                        for key in sqldict:
                            if len(sqldict['Name'])>len(sqldict[key]):
                                sqldict[key].append('')
                
                    # condition to start recording the names
                    if tp[0] == tp[0] and tp[0] == 'Name' and reg.split()[-1]!='21':
                        gate = True
                    elif tp[0] == tp[0] and tp[0] == 'Company name' and reg.split()[-1]=='21':
                        gate = True
            
        

    elif '.pdf' in url.lower():
        driver.get(url)
        sleep(3)
        
        while len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele])==0:
            print('Waiting for file to download')
            sleep(2)
    
        print(os.listdir(tempfolder))
        pdf_file = os.listdir(tempfolder)[0]
        filePath = os.path.join(tempfolder, pdf_file)
    
        tables = camelot.read_pdf(filePath, pages='all', flavor='stream', edge_tol=50)
        street = ''
        code_postal = ''
        name = ''
        address = ''
    
        # Iterate over the tables of each pages
        for i in range(1,tables.n):
            df_Table = tables[i].df

            for j in range(1,len(df_Table)):
                if len(pattern.findall(df_Table[0][j])) > 0:
                    if name.strip() != '':
                        address = street.strip() + ', ' + code_postal.strip()
                        #print('Name: ',name.strip())
                        #print('Address: ',address,'\n')
                        sqldict['Name'].append(name.strip())
                        sqldict['Address_1'].append(address)
                        sqldict['RegCtry'].append('LI')
                        sqldict['Cntry'].append('LI')
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['ListCode'].append('9')
                        sqldict['RegCode'].append('FMALI')
                        sqldict['RegulationType'].append('Regulated')
                        sqldict['ListName'].append(Typology[reg])
                    
                        # Fill the rest with empty string
                        for key in sqldict.keys():
                            if len(sqldict['Name']) > len(sqldict[key]):
                                sqldict[key].append('')
                        name = ''
                        street = ''
                        code_postal = ''
                    else:
                        name = df_Table[1][j].strip()

                if df_Table[0][j].strip() not in ['Nr.'] and df_Table[1][j].strip() != 'Total: 35':
                    name = name + ' ' + df_Table[1][j].strip()
                    street = street + ' ' + df_Table[2][j].strip()
                    code_postal = code_postal + ' ' + df_Table[3][j].strip()
            
        if name != '':
            address = street.strip() + ', ' + code_postal.strip()
            #print('Name: ', name.strip())
            #print('Address: ',address)
            sqldict['Name'].append(name.strip())
            sqldict['Address_1'].append(address)
            sqldict['RegCtry'].append('LI')
            sqldict['Cntry'].append('LI')
            sqldict['ListProcessDate'].append(processdate)
            sqldict['ListCode'].append('9')
            sqldict['RegCode'].append('FMALI')
            sqldict['RegulationType'].append('Regulated')
            sqldict['ListName'].append(Typology[reg])
            # Fill the rest with empty string
            for key in sqldict.keys():
                if len(sqldict['Name']) > len(sqldict[key]):
                    sqldict[key].append('')
        
        
        os.remove(filePath)


Working with CH FINMA 1
['beh.xlsx']
Working with CH FINMA 2
['raiff.xlsx']
Working with CH FINMA 3
['repbeh.xlsx']
Working with CH FINMA 4
['beh_status2.xlsx']
Working with CH FINMA 5
['vu.xlsx']
Working with CH FINMA 6
['vk.xlsx']
Working with CH FINMA 7
['fintech.xlsx']
Working with CH FINMA 8
CH FINMA 8 - DISREGARD
Working with CH FINMA 9
['afch.xlsx']
Working with CH FINMA 10
['afetr.xlsx']
Working with CH FINMA 11
['flvervt.xlsx']
Working with CH FINMA 12
CH FINMA 12 - DISREGARD
Working with CH FINMA 13
CH FINMA 13 - DISREGARD
Working with CH FINMA 14
CH FINMA 14 - DISREGARD
Working with CH FINMA 16
['vvtr.xlsx']
Working with CH FINMA 17
['regst.xlsx']
Working with CH FINMA 18
['repvkv.xlsx']
Working with CH FINMA 19
['repvvtr.xlsx']
Working with CH FINMA 20
['prprosp.xlsx']
Working with CH FINMA 21
['sro.xlsx']
Working with CH FINMA 22
['ao.xlsx']


In [7]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)



C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_22392\2044201188.py:9: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [8]:
df.to_csv('total_2025.csv')

In [ ]:
df.to_csv('total_2025_chfinma.csv')

In [ ]:
df.to_csv('list7_11_11_4_2025.csv')

In [ ]:
df_excel = pd.read_excel(filePath, engine='openpyxl', header=None, dtype=str)
df_excel.fillna('*nan*', inplace=True)

In [ ]:
gate = False
city_in_column_0 = False
city_in_tuple_idx1 = False
city_in_tuple_idx2 = False

list_of_tuples = df_excel.to_numpy()#list_of_tuples = df_excel.index.values.tolist()
if 'City' in df_excel[0].tolist():
    city_in_column_0 = True
elif 'City' in [tuple(item)[1] for item in list_of_tuples]:
    city_in_tuple_idx1 = True
elif 'City' in [tuple(item)[2] for item in list_of_tuples]:
    city_in_tuple_idx2 = True

for i, tp in enumerate (list_of_tuples):
    
    nan_count = sum(x == '*nan*' for x in tp)
    if nan_count+1>=len(tp) and reg.split()[-1] != '6':
        # print(tp)
        # print(len(tp))
        pass
    elif reg.split()[-1] == '6' and nan_count>=len(tp):
        pass
    else:
    # cast tp to tuple because of the python version!!
        tp = tuple(tp)
        print(tp)
        
    
        # condition to stop recording the names
        if gate and not tp[0] == tp[0] or '*nan*' in tp[0].strip() :
            gate = False
        
    
        # the gate is open. Record the names
        if gate and tp[0].strip() != '' :
            pass
            print('Name: ',tp[0].strip())
                        # condition to start recording the names
    if tp[0] == tp[0] and tp[0] == 'Name':
        
        gate = True


('Company name', '*nan*', 'Address', 'Tel. / fax', 'E-mail', '*nan*', '*nan*', 'Homepage')
('AOOS - Schweizerische Aktiengesellschaft für Aufsicht', '*nan*', 'Clausiusstrasse 50\n8006 Zürich', '+41 (44) 215 98 98\n', 'info@aoos.ch', '*nan*', '*nan*', 'https://www.aoos.ch')
('ASSOCIATION ROMANDE DES INTERMEDIAIRES FINANCIERS (ARIF)', '*nan*', 'Rue de Rive 8\nCase postale\n1211 Genève 3', '+41 (0)22 310 07 35\n+41 (0)22 310 07 39', 'info@arif.ch', '*nan*', '*nan*', 'https://www.arif.ch')
('Organismo di Autodisciplina dei Fiduciari del Cantone Ticino (OAD FCT)', '*nan*', 'Piazza Cioccaro 7\nCasella postale\n6901 Lugano', '+41 (0)91 923 98 14\n+41 (0)91 922 94 40', 'segretariato@oadfct.ch', '*nan*', '*nan*', 'https://www.oadfct.ch')
('PolyReg Allg. Selbstregulierungs-Verein', '*nan*', 'Florastrasse 44\n8008 Zürich', '+41 (43) 488 52 80\n+41 (43) 499 80 56', 'info@polyreg.ch', '*nan*', '*nan*', 'https://www.polyreg.ch')
('Schweizerischer Leasingverband (SRO SLV)', '*nan*', 'Rämistrasse 5\nP

In [ ]:
# df_excel = pd.read_excel(filePath, engine='openpyxl', header=None, dtype=str)
# df_excel.fillna('*nan*', inplace=True)
# for _, row in df_excel.iloc[:,:2].iterrows:
#     #print(row.values)
#     if '*nan*' not in row.values:
#         print(row.values)



In [ ]:
# import numpy as np
# valid_df = df_excel.iloc[:,:2]
# for _, row in valid_df.iterrows():
#     #print(row.values)
#     if '*nan*' not in row.values:
#         print(row.values)


['Name' 'City']
['Aargauische Kantonalbank' 'Aarau 1']
['ABANCA CORPORACION BANCARIA S.A., Betanzos, succursale de Genève'
 'Genève 1']
['acrevis Bank AG' 'St. Gallen']
['AEK BANK 1826 Genossenschaft' 'Thun']
['AFS Execution Services B.V., Amsterdam, Zweigniederlassung Zürich'
 'Zürich']
['Allfunds Bank S.A., Madrid, Zurich Branch' 'Zürich']
['Alpha RHEINTAL Bank AG' 'Heerbrugg']
['Alpian SA' 'Genève']
['Alternative Bank Schweiz AG' 'Olten']
['AMINA Bank AG' 'Zug']
['Appenzeller Kantonalbank' 'Appenzell']
['Aquila AG' 'Zürich']
['Arab Bank (Switzerland) Ltd.' 'Genève 3']
['AXION SWISS BANK SA' 'Lugano']
['Baader Helvea AG' 'Zürich']
['Baloise Bank AG' 'Solothurn']
['Banca Aletti & C. (Suisse) SA' 'Lugano']
['Banca Credinvest SA' 'Lugano']
['BANCA DEL CERESIO SA' 'Lugano']
['BANCA DEL SEMPIONE SA' 'Lugano']
['Banca dello Stato del Cantone Ticino' 'Bellinzona']
['Banca Popolare di Sondrio (Suisse) SA' 'Lugano']
['BANCA ZARATTINI & CO. SA' 'Lugano']
['Banco Itaú (Suisse) SA' 'Zürich']
['B

In [ ]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)



C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_12320\2044201188.py:9: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [ ]:
df.to_csv('ch_finma_total.csv')